In [ ]:
### forest analysis 2 --
### need 'd2' for python environment
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
from osgeo import gdal
from shapely.geometry import MultiPolygon
from shapely.ops import unary_union
import csv
import sys

In [ ]:
### load... EVT metadata
evt_metadata_cats = {}
with open("../data/raw/raw_raster/src/LF2022_EVT_230_CONUS/CSV_Data/LF22_EVT_230.csv") as csv_file:
    csv_read=csv.reader(csv_file, delimiter=',')
    firstflag = False
    for row in csv_read:
        #print(row)
        if firstflag:
            evt_metadata_cats[int(row[0])] = [row[1], float(row[13])/255, float(row[14])/255, float(row[15])/255]
        else:
            firstflag = True

### test... print metadata 
for k in evt_metadata_cats:
    print(k, evt_metadata_cats[k])

In [ ]:
### need to load the output file of geoplotting.py
### need to figure out whats going on here probably
comps = np.load('../visualize/geographic/comparicomputations.npy')
print("loaded")

### need to split, for each plot, into evt segments
### uniqueset is ...
uniqueset = {}
for i in range(len(comps)):
    if comps[i][0] not in uniqueset:
        uniqueset[comps[i][0]] = [[], evt_metadata_cats[comps[i][0]]]
    else:
        uniqueset[comps[i][0]][0].append(comps[i])

print(len(uniqueset.keys()))
evt_keys = list(uniqueset.keys())
for colork in evt_keys:
    uniqueset[colork][0] = np.array(uniqueset[colork][0])
    print(uniqueset[colork][0].shape)

In [ ]:
freq = []
names = []
print(len(comps))
for colork in evt_keys:
    #print(uniqueset[colork][1][0], len(uniqueset[colork][0])/ len(comps))
    freq.append(len(uniqueset[colork][0]))
    names.append(uniqueset[colork][1][0])

freq = np.array(freq)
argsrted = np.argsort(-freq)
sumpct = 0
for i in range(len(freq)):
    print(names[argsrted[i]], freq[argsrted[i]] / len(comps))
    sumpct += freq[argsrted[i]]
    if i == 7:
        print(sumpct / len(comps))
    if i == 12:
        print(sumpct / len(comps))

reduced_keys = []
reduced_names = []
reduced_pct = []
#reduced_set = {}
for i in range(8):
    reduced_names.append(names[argsrted[i]])
    reduced_keys.append(evt_keys[argsrted[i]])
    reduced_pct.append(float(freq[argsrted[i]] / len(comps)))
#reduced_pct = np.array(reduced_pct).astype(float)

In [ ]:
for i in range(len(reduced_pct)):
    reduced_pct[i] *= 100

In [ ]:
plt.barh(reduced_names, reduced_pct)
plt.xlabel("Percent of Total Samples")
plt.title("Existing Vegetation Type Frequency (Top 8)")

In [ ]:
### need to load in all the rasters and make nice images out of them jesus
geoperf_locs = ["../visualize/geographic/c2_late_a_pyramid_error_1_wue.tif",
                "../visualize/geographic/c2_late_a_pyramid_predictions_1_wue.tif",
                "../visualize/geographic/c2_late_a_pyramid_error_1_esi.tif",
                "../visualize/geographic/c2_late_a_pyramid_predictions_1_esi.tif",
                "../visualize/geographic/c2_late_a_pyramid_error_1_agb.tif",
                "../visualize/geographic/c2_late_a_pyramid_predictions_1_agb.tif"]
                
geoperf_rasters = []
geoperf_ndv = []

for gploc in geoperf_locs:
    print("loading from", gploc)
    gpraster = gdal.Open(gploc)
    gpband = gpraster.GetRasterBand(1)

    geoperf_ndv.append(gpband.GetNoDataValue())
    geoperf_rasters.append(np.array(gpraster.ReadAsArray().transpose()))
    print(" -> read array", geoperf_rasters[-1].shape)
    
    ### trim to remove extra ndv ...
    tempndv = np.argwhere(geoperf_rasters[-1] != geoperf_ndv[-1])
    x_min, y_min = tempndv.min(axis=0)
    x_max, y_max = tempndv.max(axis=0)
    geoperf_rasters[-1] = geoperf_rasters[-1][x_min:x_max+1, y_min:y_max+1]
    print(" -> cropped to", geoperf_rasters[-1].shape)

combined_min = np.genfromtxt("../data/ml_sets/pyramid_v10/norm_layer_mins_combined.csv", delimiter=',')
combined_max = np.genfromtxt( "../data/ml_sets/pyramid_v10/norm_layer_maxs_combined.csv", delimiter=',')
print(len(combined_min), len(combined_max))
combined_ids2 = {1: 15, 
                 0: 15, 
                 3: 16, 
                 2: 16, 
                 4: 17, 
                 5: 17}
for ii in combined_ids:
    print(combined_min[combined_ids[ii]], combined_max[combined_ids[ii]])


for ii in range(len(geoperf_rasters)//2):
    ### actually, as far as error goes...
    ### if x in [0, 1] and error is (x1 - x2)^2
    ### now scale to [a, b]: error scales with (a-b)^2
    geoperf_rasters[(ii*2) + 1] = (geoperf_rasters[(ii*2) + 1] * (combined_max[combined_ids2[(ii*2) + 1]] - combined_min[combined_ids2[(ii*2) + 1]])) + combined_min[combined_ids2[(ii*2) + 1]]
    print(" -> scaled to", combined_min[combined_ids2[(ii*2)]], combined_max[combined_ids2[(ii*2)]])
    geoperf_rasters[(ii*2)] = geoperf_rasters[(ii*2)] * ((combined_max[combined_ids2[(ii*2)]] - combined_min[combined_ids2[(ii*2)]]) ** 2)


In [ ]:
plt_titles = ["Late-Combination WUE Error",
              "Late-Combination WUE Predictions",
              "Late-Combination ESI Error",
              "Late-Combination ESI Predictions",
              "Late-Combination AGB Error",
              "Late-Combination AGB Predictions"]

plt_save = ["wue_error",
            "wue_predictions",
            "esi_error",
            "esi_predictions",
            "agb_error",
            "agb_predictions"]
#nice_y_names = ["Predicted Water Use Efficiency \n (g C kg$\mathregular{^{-1}}$ H$\mathregular{_2}$O)", 
#                "Predicted Evaporative Stress Index", "Predicted Above Ground Biomass \n (Mg ha$\mathregular{^{-1}}$)"]

plt_units = ["WUE MSE", "WUE Predictions (g C kg$\mathregular{^{-1}}$ H$\mathregular{_2}$O)", 
             "ESI MSE", "ESI Predictions", 
             "AGB MSE", "AGB Predictions (Mg ha$\mathregular{^{-1}}$)"]

textlabels = ["B", "A", "D", "C", "F", "E"]
labelpad = [-46, -45, -52, -45, -62, -60]

for i in range(len(geoperf_rasters)):
    #fig, ax = plt.subplots()
    plt.imshow(geoperf_rasters[i].transpose(), cmap='plasma_r', interpolation='nearest')
    plt.axis('off')
    plt.annotate(textlabels[i], xy=(0.85, 1), xycoords='axes fraction', xytext=(+0.5, -0.5), textcoords='offset fontsize',
        fontsize='medium', verticalalignment='top', horizontalalignment='right', fontfamily='serif',
        bbox=dict(facecolor='0.9', edgecolor='none', pad=3.0))
    #plt.title(plt_titles[i])
    cbar = plt.colorbar(fraction=0.025, pad=-0.15)
    cbar.set_label(plt_units[i], rotation=90, labelpad=labelpad[i])
    plt.savefig("../visualize/realgeoplots/" + plt_save[i] + ".png", bbox_inches = 'tight')
    plt.close()
    

In [ ]:
nine_keys = []
nine_names = []
nine_pct = []
#reduced_set = {}
for i in range(9):
    nine_names.append(names[argsrted[i]])
    nine_keys.append(evt_keys[argsrted[i]])
    nine_pct.append(float(freq[argsrted[i]] / len(comps)))

In [ ]:
for i in range(9):
    print(nine_keys[i])
    #print(uniqueset[nine_keys[i]][1][0])

reorder9 = [7055.0, 7011.0, 7050.0, 7107.0, 7054.0, 7051.0, 7059.0, 7016.0]
print(reorder9)

In [ ]:
xkeys = ["lf_cc", "lf_evc", "standage"]
ykeys = ["yhat_wue", "yhat_esi", "yhat_agb"]
nice_x_names = ["Landfire Canopy Cover (Percent)", "Landfire Existing Vegetation Cover (Percent)", "Forest Stand Age (Years)"]
nice_y_names = ["Predicted Water Use Efficiency \n (g C kg$\mathregular{^{-1}}$ H$\mathregular{_2}$O)", 
                "Predicted Evaporative Stress Index", "Predicted Above Ground Biomass \n (Mg ha$\mathregular{^{-1}}$)"]
nice_x_title = ["Landfire Canopy Cover ", "Landfire Existing Vegetation Cover", "Forest Stand Age"]
#letters = ["A", "B", "C", "D", "E", "F", "G", "H", "I"]
letters = ["A", "D", "G", "B", "E", "H", "C", "F", "I"]
letterctr = 0
## (g C kg$\mathregular{^-1}$ H $\mathregular{_2}$ O)
## ($g C kg^{-1} H_2O$)

ngrid = [2, 4]
colorsm = plt.cm.get_cmap('hsv', 9)

for xkey in range(len(xkeys)):
    for ykey in range(len(ykeys)):
        fig, axs = plt.subplots(ngrid[0]+1, ngrid[1]+1, layout='constrained', figsize=(10, 6), gridspec_kw = {'wspace': 0, 'hspace':0})#, sharex=True, sharey=True)

        ax_histxs = []
        ax_histys = []
        
        gridhisty = [[] for ii in range(ngrid[0])]
        gridhistx = [[] for ii in range(ngrid[1])]
        gridcoloy = [[] for ii in range(ngrid[0])]
        gridcolox = [[] for ii in range(ngrid[1])]
        x_max = [[] for ii in range(ngrid[1])]
        y_max = [[] for ii in range(ngrid[0])]
        gridctr = 0
        ### iterate over rows?
        for i in range(ngrid[0]):
            ### iterate over cols
            for j in range(ngrid[1]):
                if True:
                    axs[i+1, j].scatter(uniqueset[reorder9[gridctr]][0][:,xkey+4], uniqueset[reorder9[gridctr]][0][:,ykey+1], s=0.01, 
                                alpha=0.2, #label=uniqueset[reorder9[gridctr]][1][0], 
                                color=colorsm(gridctr))
                    ### for legend
                    axs[0, ngrid[1]].scatter(uniqueset[reorder9[gridctr]][0][:1,xkey+4], uniqueset[reorder9[gridctr]][0][:1,ykey+1], s=0.001, 
                                label=uniqueset[reorder9[gridctr]][1][0], 
                                color=colorsm(gridctr))
                    
                gridhistx[j].append(uniqueset[reorder9[gridctr]][0][:,xkey+4])
                gridhisty[i].append(uniqueset[reorder9[gridctr]][0][:,ykey+1])

                gridcolox[j].append(colorsm(gridctr))
                gridcoloy[i].append(colorsm(gridctr))

                ### this is one for each on x axis (4)... add one for each y in this x position
                x_max[j].append(np.nanmax(uniqueset[reorder9[gridctr]][0][:,xkey+4]))
                ### this is one for each on y axis (2)
                y_max[i].append(np.nanmax(uniqueset[reorder9[gridctr]][0][:,ykey+1]))

                if j != 0:
                    axs[i+1,j].set_yticks([])
                if i+1 != ngrid[0]:
                    axs[i+1,j].set_xticks([])
                #axs[i+1, j].legend(loc='lower left', bbox_to_anchor=(1.02 + (i/10), 1.02), markerscale=10)
                ##loc='lower left', bbox_to_anchor=(1.02, 1.02), prop={'size': 5},markerscale=10
                gridctr += 1
        #print(x_max)
        #print(y_max)

        #axs[0, ngrid[1]].text(0, 0, letters[letterctr], fontsize=40,
        #verticalalignment='bottom', horizontalalignment='left')#, bbox=props)
        
        for i in range(ngrid[0]):
            supermax = max(y_max[i])
            superdiff = supermax * 0.05
            for j in range(ngrid[1]):
                axs[i+1, j].set_ybound(-superdiff, supermax + superdiff)
        for j in range(ngrid[1]):
            supermax = max(x_max[j])
            superdiff = supermax * 0.05
            for i in range(ngrid[0]):
                axs[i+1, j].set_xbound(-superdiff, supermax+superdiff)
        #x_min = []
        #y_min = []
        x_max = []
        y_max = []
        for ng in range(ngrid[1]):
            (ngnx, ngbinsx, ngpatchesx) = axs[0, ng].hist(gridhistx[ng], nbins_each, histtype='bar', stacked=True, log=False, color=gridcolox[ng]) #density=True
            x_max.append(np.max(ngnx))
            axs[0, ng].set_xticks([])
            if ng != 0:
                axs[0, ng].set_yticks([])
        for ng in range(ngrid[0]):
            (ngny, ngbinsy, ngpatchesy) = axs[ng+1, ngrid[1]].hist(gridhisty[ng], nbins_each, histtype='bar', stacked=True, log=False, orientation='horizontal', color=gridcoloy[ng])
            y_max.append(np.max(ngny))
            axs[ng+1, ngrid[1]].set_yticks([])
            if ng+1 != ngrid[0]:
                axs[ng+1, ngrid[1]].set_xticks([])
        true_y_max = max(y_max)
        true_x_max = max(x_max)
        #print(max(x_max), max(y_max))
        for ng in range(ngrid[0]):
            axs[ng+1, ngrid[1]].set_xbound(0, true_y_max)
        for ng in range(ngrid[1]):
            axs[0, ng].set_ybound(0, true_x_max)
        
        axs[ngrid[0], 1].set_xlabel(nice_x_names[xkey])
        axs[1, 0].set_ylabel(nice_y_names[ykey])
        axs[0, 0].set_ylabel("Frequency")
        axs[ngrid[0], ngrid[1]].set_xlabel("Frequency")

        fig.legend(loc='upper left', bbox_to_anchor=(0, -0.01), ncol=2, markerscale=100, prop={'size': 8})
        axs[0, ngrid[1]].annotate(letters[letterctr], xy=(0.2, 0.2), xycoords='axes fraction', textcoords='offset fontsize',
        fontsize='large', verticalalignment='bottom', horizontalalignment='left', fontfamily='serif',
        bbox=dict(facecolor='0.9', edgecolor='none', pad=3.0))
        #axs[0, ngrid[1]].legend(loc='lower right', bbox_to_anchor=(0.05, 1.05), fancybox=True, shadow=True, 
        #                        ncol=2, markerscale=10, prop={'size': 5})
        #axs[0, ngrid[1]].legend(loc='lower left', bbox_to_anchor=(0, 0), markerscale=20, prop={'size': 5})
        axs[0, ngrid[1]].axis('off')
        #axs[0, 3].set_title("Late-Combination CNN " + nice_y_names[ykey] + " Versus " + nice_x_title[xkey] + "\nOver Landfire Vegetation Types")

        plt.savefig("../visualize/covars/8box_" + xkeys[xkey] + "_" + ykeys[ykey] + ".png", bbox_inches = 'tight')
        letterctr += 1
        break
    break
